In [1]:
# Fairness Audit - Active Learning
# Goal: Compare whether equitable AL improved performance
# across medical specialties compared to standard AL

import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report
from fairlearn.metrics import MetricFrame, equalized_odds_difference, demographic_parity_difference
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Libraries loaded")

Libraries loaded


In [2]:
# Load data
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")
label_map = pd.read_csv("../data/processed/label_map.csv")
num_labels = len(label_map)

# Use small test subset for speed
test_texts = test_df['text'].iloc[:50].tolist()
test_labels = test_df['label'].iloc[:50].tolist()

print("Data loaded")

Data loaded


In [3]:
# Dataset class and tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class ClinicalNotesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("Dataset class defined")

Dataset class defined


In [ ]:
# Train two final models
# 1. Standard AL: random sampling (baseline)
# 2. Equitable AL: equity-constrained sampling

def train_final_model(texts, labels, num_labels, epochs=1):
    dataset = ClinicalNotesDataset(texts, labels, tokenizer)
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    model.train()
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels_batch)
        outputs.loss.backward()
        optimizer.step()
    return model

np.random.seed(42)
subset_size = 150

# Standard: random sample
random_idx = np.random.choice(len(train_df), subset_size, replace=False)
std_texts = train_df['text'].iloc[random_idx].tolist()
std_labels = train_df['label'].iloc[random_idx].tolist()

# Equitable: balanced sample across specialties
eq_texts, eq_labels = [], []
per_class = subset_size // num_labels
for label in range(num_labels):
    class_df = train_df[train_df['label'] == label].head(per_class)
    eq_texts.extend(class_df['text'].tolist())
    eq_labels.extend(class_df['label'].tolist())

print("Training standard model...")
std_model = train_final_model(std_texts, std_labels, num_labels)
print("Training equitable model...")
eq_model = train_final_model(eq_texts, eq_labels, num_labels)
print("Both models trained")

Training standard model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training equitable model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
